# CoDA-GQA-L: Mistral-7B Benchmarking & Ablations

**Target hardware**: H100 80GB (RunPod)

This notebook runs the full evaluation pipeline for the trained Mistral-7B-v0.3 model:

1. **Setup**: Install deps, clone repo, pull trained checkpoint from HuggingFace
2. **Smoke test**: Forward equivalence on SmolLM2-135M (fast sanity check)
3. **Cold-swap PPL**: Baseline PPL before any training (Table 1)
4. **Trained model PPL**: Load Phase 2 checkpoint, eval at multiple context lengths (Table 2)
5. **Dynamic expansion PPL**: Re-eval with expansion enabled (64→128 slots)
6. **Throughput benchmarks**: tok/s prefill + decode, KV cache memory (Table 3)
7. **Ablation: diff attn contribution**: Train GQA+bounded (no diff attn) head-to-head
8. **Ablation: memory bank configs**: Sweep window-only / tiny / medium / large
9. **Memory bank metrics**: Hit rates, fill ratios, eviction counts
10. **Collect results**: Render tables, save JSONs

**Estimated time**: ~30 min without retraining, ~4-6 hrs with ablation training

## 0. Setup

In [ ]:
# Clone repo and install
!cd /workspace && git clone https://github.com/anthony-maio/CoDA-GQA-L.git 2>/dev/null || (cd /workspace/CoDA-GQA-L && git pull)
%cd /workspace/CoDA-GQA-L
!pip install -e . -q
!pip install transformers accelerate datasets huggingface_hub triton -q

In [ ]:
# Environment check
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

try:
    import triton
    print(f"Triton: {triton.__version__}")
except ImportError:
    print("Triton: NOT INSTALLED")

# Kernel availability (now subpackages of coda_gqa_l)
from coda_gqa_l.triton_bank_routing import diagnose as _br_diag
print(_br_diag())
from coda_gqa_l.triton_diff_flash import diagnose as _df_diag
print(_df_diag())

In [ ]:
# Results directory
import os
from datetime import datetime

RESULTS_DIR = f"/workspace/results_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results: {RESULTS_DIR}")

In [ ]:
# Run the test suite to make sure everything is working
!python -m pytest tests/ -q --tb=short

## 1. Smoke Test (Forward Equivalence)

In [ ]:
# Quick forward-check on SmolLM2-135M: proves weight mapping is lossless
!python benchmarks/eval_llm.py \
    --model HuggingFaceTB/SmolLM2-135M \
    --experiment forward-check \
    --dtype fp32 \
    --results-dir {RESULTS_DIR}

## 2. Cold-Swap Perplexity (Table 1 — No Training Needed)

Swaps Mistral-7B attention layers to CoDA-GQA-L with zero-init params (lambda≈0, theta=0).
This measures the structural overhead of the swap before any training.

**Use fp32** for cold-swap — bf16 rounding compounds through 32 layers and inflates PPL artificially.

In [ ]:
!python benchmarks/eval_llm.py \
    --model mistralai/Mistral-7B-v0.3 \
    --experiment perplexity \
    --dtype fp32 \
    --results-dir {RESULTS_DIR}

## 3. Trained Model Evaluation (Table 2)

Load the Phase 2 bounded checkpoint from HuggingFace and evaluate PPL.

The checkpoint was trained:
- Phase 1: 400 steps unbounded (PPL 5.94)
- Phase 2: 600 steps bounded W=256/Me=64/Ms=64 (PPL 5.81)

### Download checkpoint

In [ ]:
from huggingface_hub import hf_hub_download
import os

CODA_REPO = "anthonym21/Mistral-7B-v0.3-CoDA-GQA-L"
ADAPTER_FILE = "coda_adapters.pt"

adapter_path = hf_hub_download(repo_id=CODA_REPO, filename=ADAPTER_FILE)
print(f"Adapter weights: {adapter_path}")

# Verify it's Phase 2 (should have write_proj keys)
import torch
state = torch.load(adapter_path, map_location="cpu", weights_only=True)
layer0_keys = list(state[0].keys()) if isinstance(state, list) else list(state.keys())
has_write_proj = any("write_proj" in k for k in layer0_keys)
param_count = sum(v.numel() for d in (state if isinstance(state, list) else [state]) for v in d.values())
print(f"Phase 2 checkpoint: {has_write_proj} ({param_count:,} params)")
if not has_write_proj:
    print("WARNING: This looks like Phase 1 weights (no write_proj). PPL may be higher than expected.")
del state

In [ ]:
# Evaluate trained model — medium config (W=256, Me=64, Ms=64)
!python benchmarks/eval_llm.py \
    --model mistralai/Mistral-7B-v0.3 \
    --experiment perplexity \
    --adapter-weights {adapter_path} \
    --head-norm-mode identity \
    --dtype bf16 \
    --results-dir {RESULTS_DIR}

## 4. Dynamic Expansion Evaluation

Test whether expanding banks from 64→128 slots improves PPL at long contexts.
Uses the same trained checkpoint — expansion is an inference-time feature.

This is a quick eval — no retraining needed.

In [ ]:
# Expansion eval: compare fixed vs expanding banks at 4096 and 8192 context
# We do this programmatically since eval_llm.py doesn't yet have --max-landmarks flags

import copy
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from coda_gqa_l import LlamaCoDAAdapter, CoDAGQALandmarkPerf2

MODEL_ID = "mistralai/Mistral-7B-v0.3"
device = torch.device("cuda")
dtype = torch.bfloat16

print("Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="auto")
model.eval()

# Load dataset
print("Loading WikiText-2...")
ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
text = "\n\n".join([t for t in ds["text"] if t.strip()])
tokens = tokenizer.encode(text)
print(f"  Total tokens: {len(tokens):,}")

In [ ]:
def eval_bounded_ppl(model_orig, tokens, adapter_path, *,
                     window=256, Me=64, Ms=64, max_Me=None, max_Ms=None,
                     context_lengths=[512, 1024, 2048, 4096, 8192],
                     max_tokens=50000, block_size=256):
    """Evaluate bounded PPL at multiple context lengths."""
    import torch.nn.functional as F
    from pathlib import Path

    results = {}
    for ctx_len in context_lengths:
        model_b = copy.deepcopy(model_orig)
        adapters = LlamaCoDAAdapter.swap_llama_layers(
            model_b, bounded=True,
            window=window, num_landmarks_exact=Me, num_landmarks_summary=Ms,
            max_landmarks_exact=max_Me, max_landmarks_summary=max_Ms,
            block_size=block_size,
            head_norm_mode="identity",
        )

        # Load adapter weights
        state = torch.load(adapter_path, map_location="cpu", weights_only=True)
        for i, adapter in enumerate(adapters):
            if i < len(state):
                adapter.load_state_dict(state[i], strict=False)
        del state

        model_b.eval()

        # Streaming PPL over non-overlapping chunks
        total_loss = 0.0
        total_tokens_scored = 0
        offset = 0
        tok_tensor = torch.tensor(tokens[:max_tokens], dtype=torch.long)

        while offset + ctx_len <= len(tok_tensor):
            chunk = tok_tensor[offset:offset + ctx_len].unsqueeze(0).to(device)

            # Reset state between independent chunks
            for adapter in adapters:
                if hasattr(adapter, "reset_state"):
                    adapter.reset_state()

            with torch.no_grad():
                out = model_b(input_ids=chunk)
                logits = out.logits[:, :-1, :].float()
                labels = chunk[:, 1:]
                loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                                      labels.reshape(-1), reduction="sum")
                total_loss += loss.item()
                total_tokens_scored += labels.numel()

            offset += ctx_len

        ppl = float("inf")
        if total_tokens_scored > 0:
            import math
            ppl = math.exp(total_loss / total_tokens_scored)

        results[ctx_len] = {"ppl": round(ppl, 2), "tokens_scored": total_tokens_scored}
        print(f"  ctx={ctx_len:>5d}  PPL={ppl:.2f}  ({total_tokens_scored:,} tokens)")

        del model_b
        torch.cuda.empty_cache()

    return results

In [ ]:
# Fixed banks (current default)
print("=" * 60)
print("Fixed banks: W=256, Me=64, Ms=64 (384 total)")
print("=" * 60)
ppl_fixed = eval_bounded_ppl(model, tokens, adapter_path,
                             window=256, Me=64, Ms=64)

In [ ]:
# Dynamic expansion (64→128 per bank)
print("=" * 60)
print("Dynamic expansion: W=256, Me=64→128, Ms=64→128 (384→512)")
print("=" * 60)
ppl_expand = eval_bounded_ppl(model, tokens, adapter_path,
                              window=256, Me=64, Ms=64,
                              max_Me=128, max_Ms=128)

In [ ]:
# Compare
print("\n" + "=" * 60)
print("Expansion Impact")
print("=" * 60)
print(f"{'Context':>8}  {'Fixed':>8}  {'Expand':>8}  {'Delta':>8}  {'Improv':>8}")
print("-" * 50)
for ctx in sorted(ppl_fixed.keys()):
    f = ppl_fixed[ctx]["ppl"]
    e = ppl_expand[ctx]["ppl"]
    delta = f - e
    pct = (delta / f) * 100 if f > 0 else 0
    print(f"{ctx:>8d}  {f:>8.2f}  {e:>8.2f}  {delta:>+8.2f}  {pct:>+7.1f}%")

# Save
expansion_results = {"fixed": ppl_fixed, "expansion": ppl_expand}
with open(f"{RESULTS_DIR}/expansion_comparison.json", "w") as f:
    json.dump(expansion_results, f, indent=2)
print(f"\nSaved to {RESULTS_DIR}/expansion_comparison.json")

## 5. Throughput Benchmarks (Table 3)

In [ ]:
!python benchmarks/run_suite.py --results-dir {RESULTS_DIR}
!python benchmarks/render_tables.py --results-dir {RESULTS_DIR}

In [ ]:
# Display the tables
from pathlib import Path
tables = Path(RESULTS_DIR) / "tables.md"
if tables.exists():
    from IPython.display import Markdown
    display(Markdown(tables.read_text()))
else:
    print("Tables not generated yet")

## 6. Ablation: Differential Attention Contribution

**Key question**: Does differential attention actually help bounded inference, or would standard GQA with the same memory banks achieve similar PPL?

Two training runs head-to-head:
1. **GQA + bounded** (`--no-differential`): standard GQA with bounded cache
2. **CoDA + bounded**: full differential attention with bounded cache

Previous results: CoDA=5.81, GQA=5.97 (~2.7% improvement). Re-running for verification.

**This takes ~4 hours. Skip if you've already run this.**

In [ ]:
RUN_ABLATION_TRAINING = False  # Set to True to run (~4 hours)

if RUN_ABLATION_TRAINING:
    print("Running ablation training...")
    !bash benchmarks/run_ablation_h100.sh
else:
    print("Skipping ablation training (set RUN_ABLATION_TRAINING = True to run)")
    print("Previous results: CoDA+bounded = 5.81 PPL, GQA+bounded = 5.97 PPL")

## 7. Ablation: Memory Bank Configurations

Sweep bounded configs to measure the contribution of exact and summary banks.

Configs:
- `window-only`: W=256, Me=0, Ms=0 (just sliding window, no banks)
- `tiny`: W=128, Me=32, Ms=32 (192 total)
- `medium`: W=256, Me=64, Ms=64 (384 total — default)
- `large`: W=512, Me=128, Ms=128 (768 total)

In [ ]:
# Memory bank config sweep
!python benchmarks/eval_llm.py \
    --model mistralai/Mistral-7B-v0.3 \
    --experiment perplexity \
    --bounded-configs window-only,tiny,medium,large \
    --adapter-weights {adapter_path} \
    --head-norm-mode identity \
    --dtype bf16 \
    --results-dir {RESULTS_DIR}

## 8. Memory Bank Metrics

Collect hit rates, fill ratios, and eviction counts during bounded generation.

In [ ]:
!python examples/bounded_generate.py \
    --collect-metrics \
    --max-new-tokens 500 \
    2>&1 | tee {RESULTS_DIR}/bounded_generate_metrics.log

## 9. Needle-in-Haystack Retention

In [ ]:
!python examples/needle_demo.py 2>&1 | tee {RESULTS_DIR}/needle_demo.log
!RUN_LONG=1 python examples/needle_demo.py 2>&1 | tee -a {RESULTS_DIR}/needle_demo_long.log

## 10. Optional: Re-train with Dynamic Expansion

Train Phase 2 with `medium-expand` config (banks grow from 64→128 slots).
This teaches the model to work across the full capacity range.

**Only needed if expansion eval (Section 4) shows significant improvement.**

In [ ]:
RUN_EXPANSION_TRAINING = False  # Set to True if expansion eval looks promising

if RUN_EXPANSION_TRAINING:
    print("Training with dynamic expansion (medium-expand)...")
    !python benchmarks/train_coda.py \
        --model mistralai/Mistral-7B-v0.3 \
        --max-steps 400 \
        --bounded-steps 600 \
        --bounded-config medium-expand \
        --batch-size 1 \
        --grad-accum 8 \
        --seq-len 2048 \
        --lr 5e-5 \
        --lr-coda 1e-3 \
        --bounded-lr-scale 0.5 \
        --eval-every 200 \
        --save-every 200 \
        --output-dir {RESULTS_DIR}/expansion_training
    
    print("\nEvaluating expansion-trained model...")
    !python benchmarks/eval_llm.py \
        --model mistralai/Mistral-7B-v0.3 \
        --experiment perplexity \
        --adapter-weights {RESULTS_DIR}/expansion_training/best \
        --head-norm-mode identity \
        --dtype bf16 \
        --results-dir {RESULTS_DIR}
else:
    print("Skipping expansion training (set RUN_EXPANSION_TRAINING = True to run)")

## 11. Collect All Results

In [ ]:
import json
from pathlib import Path

results_path = Path(RESULTS_DIR)
json_files = sorted(results_path.glob("*.json"))

print(f"\nResults in {RESULTS_DIR}:")
print("=" * 60)
for f in json_files:
    size = f.stat().st_size
    print(f"  {f.name:<50s}  {size:>8,d} bytes")

log_files = sorted(results_path.glob("*.log"))
for f in log_files:
    size = f.stat().st_size
    print(f"  {f.name:<50s}  {size:>8,d} bytes")

print(f"\n  Total: {len(json_files)} JSON + {len(log_files)} logs")

In [ ]:
# Render final tables
!python benchmarks/render_tables.py --results-dir {RESULTS_DIR} 2>/dev/null || true

tables_file = Path(RESULTS_DIR) / "tables.md"
if tables_file.exists():
    from IPython.display import Markdown
    display(Markdown(tables_file.read_text()))

In [ ]:
# Tar results for download
import subprocess
tar_name = f"/workspace/coda_results_{Path(RESULTS_DIR).name}.tar.gz"
subprocess.run(["tar", "czf", tar_name, "-C", str(results_path.parent), results_path.name])
print(f"\nDownload: {tar_name}")